In [4]:
import json

with open('./datasets/mocap/dataset_v3.json', 'r') as file:
    mocap_data = json.load(file) 

with open('./datasets/yolo/dataset_v3.json', 'r') as file:
    yolo_triang_data = json.load(file) 
    
with open('./datasets/yolo/dataset_best_cameras.json', 'r') as file:
    yolo_triang_best_cameras_data = json.load(file) 

print(sequences := list(yolo_triang_data.keys()))
print(joints := list(yolo_triang_data['p1s1'][0].keys()))

['p1s1', 'p1s2', 'p1s3', 'p1s4', 'p2s1', 'p2s2', 'p2s3', 'p2s4', 'p3s1', 'p3s2', 'p3s3', 'p3s4', 'p4s1', 'p4s2', 'p4s3', 'p4s4', 'p5s1', 'p5s2', 'p5s3', 'p5s4', 'p6s1', 'p6s2', 'p6s3', 'p6s4', 'p7s1', 'p7s2', 'p7s3', 'p7s4', 'p8s1', 'p8s2', 'p8s3', 'p8s4', 'p9s1', 'p9s2', 'p9s3', 'p9s4', 'p10s1', 'p10s2', 'p10s3', 'p10s4', 'p11s1', 'p11s2', 'p11s3', 'p11s4', 'p12s1', 'p12s2', 'p12s3', 'p12s4', 'p13s1', 'p13s2', 'p13s3', 'p13s4', 'p14s1', 'p14s2', 'p14s3', 'p14s4', 'p15s1', 'p15s2', 'p15s3', 'p15s4', 'p16s1', 'p16s2', 'p16s3', 'p16s4', 'p17s1', 'p17s2', 'p17s3', 'p17s4', 'p18s1', 'p18s2', 'p18s3', 'p18s4', 'p19s1', 'p19s2', 'p19s3', 'p19s4', 'p20s1', 'p20s2', 'p20s3', 'p20s4', 'p21s1', 'p21s2', 'p21s3', 'p21s4', 'p22s1', 'p22s2', 'p22s3', 'p22s4', 'p23s1', 'p23s2', 'p23s3', 'p23s4', 'p24s1', 'p24s2', 'p24s3', 'p24s4', 'p25s1', 'p25s2', 'p25s3', 'p25s4', 'p26s1', 'p26s2', 'p26s3', 'p26s4', 'p26s5', 'p26s6', 'p26s7', 'p26s8', 'p27s1', 'p27s2', 'p27s3', 'p27s4', 'p27s5', 'p27s6', 'p27s7', 

In [5]:
import numpy as np

SCALE_MULTIPLIER = 255

errors = {aprox_type: {joint_key: [] for joint_key in joints} for aprox_type in ['triang', 'triang_best_cameras']}

for i, sequence_key in enumerate(sequences):
    for frame_mocap, frame_triang, frame_triang_best_cameras in zip(mocap_data[sequence_key], yolo_triang_data[sequence_key], yolo_triang_best_cameras_data[sequence_key]):
        for joint_key in joints:
            errors['triang'][joint_key].append(np.linalg.norm(np.array(frame_mocap[joint_key]) - np.array(frame_triang[joint_key])[[1, 2, 0]]))
            errors['triang_best_cameras'][joint_key].append(np.linalg.norm(np.array(frame_mocap[joint_key]) - np.array(frame_triang_best_cameras[joint_key])[[1, 2, 0]]))

mean_errors = {aprox_type: {} for aprox_type in ['triang', 'triang_best_cameras']}

for aprox_type in ['triang', 'triang_best_cameras']:
    for joint_key in joints:
        temp_errors = errors[aprox_type][joint_key]
        mean_error = SCALE_MULTIPLIER * sum(temp_errors)/len(temp_errors)
        mean_errors[aprox_type][joint_key] = mean_error

mean_errors

{'triang': {'lhumerus': 77.1273040462476,
  'rhumerus': 74.55431368805799,
  'lfemur': 83.0294401535641,
  'rfemur': 84.95086006401229,
  'ltibia': 72.72202545350748,
  'rtibia': 67.75637503824578,
  'lfoot': 58.81620501227455,
  'rfoot': 59.49513249255424,
  'lradius': 9449.492585070353,
  'rradius': 11518.504489183644,
  'lwrist': 23911.570273968413,
  'rwrist': 19415.742417896945},
 'triang_best_cameras': {'lhumerus': 78.14793006365035,
  'rhumerus': 77.40337337639927,
  'lfemur': 81.72028512935478,
  'rfemur': 85.45292341547402,
  'ltibia': 72.35262462134176,
  'rtibia': 67.86673351496233,
  'lfoot': 67.28214148102609,
  'rfoot': 60.68447332039179,
  'lradius': 86.83426528149313,
  'rradius': 78.95804232351212,
  'lwrist': 157.43168571510776,
  'rwrist': 136.01556004798636}}

In [6]:
len([i for i in errors["triang"]["lwrist"] if i * SCALE_MULTIPLIER > 1000])

8894

In [7]:
len(errors["triang"]["lwrist"])

18764

In [21]:
import numpy as np

print(">>>> YOLO triangulation <<<<")

for joint in joints:
    print(f"\n------------- {joint.upper()} -------------")
    arr = np.array(errors["triang"][joint]) * SCALE_MULTIPLIER
    num_of_outliers = len(arr[arr > arr.mean() + 3 * arr.std()])
    print(f"Mean: {arr.mean()}")
    print(f"Median: {np.median(arr)}")
    print(f"Standard Deviation: {arr.std()}")
    print(f"Minimum: {arr.min()}")
    print(f"Maximum: {arr.max()}")
    print(f"0.9 quantile: {np.quantile(arr, 0.9)}")
    print(f"Outliers: {num_of_outliers/arr.size}")
    print(f"Error bigger than 1m: {len(arr[arr > 1000])/arr.size}")

>>>> YOLO triangulation <<<<

------------- LHUMERUS -------------
Mean: 77.12730404624807
Median: 70.32295213007754
Standard Deviation: 29.270520737235046
Minimum: 13.298805206207874
Maximum: 207.6247186708881
0.9 quantile: 119.41869984236783
Outliers: 0.01055212108292475
Error bigger than 1m: 0.0

------------- RHUMERUS -------------
Mean: 74.5543136880579
Median: 66.63826893368355
Standard Deviation: 31.609017805799894
Minimum: 6.99814569194515
Maximum: 208.17596190109975
0.9 quantile: 121.2986073944846
Outliers: 0.006928160306970795
Error bigger than 1m: 0.0

------------- LFEMUR -------------
Mean: 83.02944015356414
Median: 74.7545746862518
Standard Deviation: 38.50111652392374
Minimum: 5.582976994707402
Maximum: 275.4522578428191
0.9 quantile: 137.48770983236707
Outliers: 0.00847367299083351
Error bigger than 1m: 0.0

------------- RFEMUR -------------
Mean: 84.95086006401161
Median: 75.40470848442156
Standard Deviation: 38.697688961333384
Minimum: 5.887873047676068
Maximum: 267.

In [20]:
import numpy as np

print(">>>> YOLO triangulation best cameras <<<<")

for joint in joints:
    print(f"\n------------- {joint.upper()} -------------")
    arr = np.array(errors["triang_best_cameras"][joint]) * SCALE_MULTIPLIER
    num_of_outliers = len(arr[arr > arr.mean() + 3 * arr.std()])
    print(f"Mean: {arr.mean()}")
    print(f"Median: {np.median(arr)}")
    print(f"Standard Deviation: {arr.std()}")
    print(f"Minimum: {arr.min()}")
    print(f"Maximum: {arr.max()}")
    print(f"0.9 quantile: {np.quantile(arr, 0.9)}")
    print(f"Outliers: {num_of_outliers/arr.size}")
    print(f"Error bigger than 1m: {len(arr[arr > 1000])/arr.size}")

>>>> YOLO triangulation best cameras <<<<

------------- LHUMERUS -------------
Mean: 78.14793006365038
Median: 71.72762873179647
Standard Deviation: 28.168421819600916
Minimum: 17.151390104530705
Maximum: 243.3040668965457
0.9 quantile: 116.0131542219917
Outliers: 0.01476231080793008
Error bigger than 1m: 0.0

------------- RHUMERUS -------------
Mean: 77.40337337639943
Median: 71.00112004656359
Standard Deviation: 29.90387596280215
Minimum: 12.014834809854053
Maximum: 240.01091828652096
0.9 quantile: 118.35322273248845
Outliers: 0.01151140481773609
Error bigger than 1m: 0.0

------------- LFEMUR -------------
Mean: 81.72028512935483
Median: 74.14156482597127
Standard Deviation: 37.28430709297499
Minimum: 1.8876017547499
Maximum: 294.4042421594425
0.9 quantile: 134.43077611836648
Outliers: 0.011617991899381794
Error bigger than 1m: 0.0

------------- RFEMUR -------------
Mean: 85.45292341547406
Median: 77.41368836711487
Standard Deviation: 37.39109480270788
Minimum: 3.5598183897891422